# Exercise 4: Pipeline ordering

*Scripts and Encoding · version 0.1.0*

> You have Kazakh text in Cyrillic and Latin, some of it diacritic-dropped, and a restoration model trained on noisy Cyrillic paired with corrected Cyrillic. Design a pipeline that preserves the original and records the script of each span. Where can that restoration model be applied, and why? How would the design change if the restoration model had been trained on Latin text instead?

The code sketches the first stages: keep the original, build an NFC view, and record script spans with offsets into that view.

**Setup.** `pip install "turkicnlp==0.5.0"`. Cells that fetch data from the web need internet access.

In [ ]:
import re
import unicodedata
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

original = "Мен Алматыда турамын. Men Astanada tūramyn. Біз казакша сөйлейміз."   # 'турамын', 'казакша': dropped letters

record = {"original": original}                                   # 1. preserve the source
record["nfc"] = unicodedata.normalize("NFC", original)            # 2. NFC processing view

# 3. script per sentence span, with character offsets into the NFC view
spans = []
for m in re.finditer(r"[^.!?]+[.!?]?", record["nfc"]):
    text = m.group().strip()
    if text:
        start = record["nfc"].index(text, m.start())
        spans.append({"start": start, "end": start + len(text), "text": text, "script": detect_script(text).name})
record["spans"] = spans
for s in spans:
    print(s)

In [ ]:
# 4. Route each span: Cyrillic spans can go to the Cyrillic restoration model;
#    Latin spans are converted to Cyrillic first (a derived view) or left as they are.
to_cyr = Transliterator("kaz", Script.LATIN, Script.CYRILLIC)

def restore_cyrillic(text):
    """Placeholder for the restoration model trained on noisy -> corrected Cyrillic."""
    return text    # e.g. would propose турамын -> тұрамын, казакша -> қазақша

for s in spans:
    if s["script"] == "CYRILLIC":
        s["restored_view"] = restore_cyrillic(s["text"])
    elif s["script"] == "LATIN":
        s["cyrillic_view"] = to_cyr.transliterate(s["text"])
        s["restored_view"] = restore_cyrillic(s["cyrillic_view"])
    print(s["script"], "|", s["text"], "->", s.get("cyrillic_view", ""), "|", s["restored_view"])

## Worked solution

**Pipeline.** (1) Store the original bytes with source and licence; decode with the known encoding. (2) Build an NFC view. (3) Identify the language (with an unrestricted model and an "unknown" option) and detect the script per span, storing offsets into the NFC view. (4) Apply language-specific repairs only to spans identified as Kazakh. (5) Restore. (6) Convert to the downstream model's script if a task comparison supports it. (7) Tokenise. Every step writes a new view and keeps a span map back to the original.

**Where the Cyrillic restoration model applies.** On Cyrillic spans, directly: it was trained on noisy and corrected Cyrillic, so that is the input distribution it knows. Latin spans can use it only after conversion to Cyrillic, and that conversion has to cope with dropped Latin diacritics (*turamyn* written without *ū*); restoration then repairs the Cyrillic view. The original Latin text stays untouched, and the converted and restored text is a derived view.

**A Latin-trained model.** The order flips: convert Cyrillic spans to Latin first (a derived view), then restore in Latin. A writer's dropped *қ* typed as *к* becomes Latin *k* after conversion, and the restoration model has to recover *q*. In both designs restoration happens in the script the model was trained on; the order follows from the tools, not from a general rule.